# Extracción de Tablas (Iniciativa LIGIE)
## Fases de Tratado
### Fase 0: Configuración General

El objetivo de este script es procesar un documento PDF oficial que contiene tablas escaneadas o incrustadas referentes a modificaciones arancelarias (Iniciativa LIGIE). Para lograr esto, se utiliza Reconocimiento Óptico de Caracteres (OCR) apoyado por el motor Tesseract.

Dependencias requeridas:
- `pandas (pd):` Manipulación de estructuras de datos y exportación a Excel.
- `img2table:` Librería especializada en detectar y extraer tablas desde PDFs/Imágenes.
- `os:` Manejo de rutas y variables de entorno del sistema.
- `re:` Expresiones regulares para limpieza de texto.

Variables Globales:
- **Rutas:** Apuntan al ejecutable de Tesseract, al archivo PDF de entrada y al archivo de salida Excel.
- **Columnas Estándar:** Define la estructura inmutable que debe tener la tabla extraída para evitar desfases de información.

In [1]:
import pandas as pd
from img2table.document import PDF
from img2table.ocr import TesseractOCR
import os
import re

# Rutas de Archivos y Dependencias
RUTA_TESSERACT = r"C:/Program Files/Tesseract-OCR"
PATH_PDF_LIGIE = r"C:/Users/Edward/Downloads/20251209-V-30-109.pdf"
PATH_SALIDA_LIGIE = "tablas_extraidas_corregido.xlsx"

# Estructura Estándar de la Tabla
COLUMNAS_ESTANDAR = [
    "CÓDIGO", 
    "DESCRIPCIÓN", 
    "UNIDAD", 
    "CUOTA_INICIATIVA", 
    "CUOTA_DICTAMEN"
]

print("--- CONFIGURACIÓN CARGADA ---")
print(f"Tesseract Path: {RUTA_TESSERACT}")
print(f"Input PDF:      {PATH_PDF_LIGIE}")
print(f"Output Excel:   {PATH_SALIDA_LIGIE}")

### Fase 0.5: Definición de Funciones

Se separan las responsabilidades lógicas del proceso de extracción y limpieza:

**1. Funciones Auxiliares (`_nombre`)**
Tareas atómicas de limpieza y configuración a nivel de fila o sistema.

- **`_configurar_entorno`**: Verifica la existencia del motor Tesseract y lo inyecta en el PATH del sistema para que `img2table` pueda utilizarlo.
- **`_limpiar_texto`**: Remueve saltos de línea indeseados y espacios adicionales en las celdas recuperadas por el OCR.
- **`_intentar_separar_cuotas`**: Aplica lógica de rescate cuando el OCR fusiona dos columnas en una sola (ej. lee "35 25" en la columna de Iniciativa y deja vacía la de Dictamen). Separa el texto basado en espacios.

**2. Funciones Intermedias (`__nombre__`)**
Procesadores que actúan a nivel de bloque de datos (DataFrames individuales).

- **`__estandarizar_tabla__`**: Recibe un bloque crudo extraído de una página, limpia sus bordes, asegura que tenga exactamente 5 columnas rellenando con nulos si faltan, le asigna los nombres de cabecera estándar y adjunta el número de página como metadato de trazabilidad.

**3. Funciones Principales (`nombre`)**
Orquestadores del flujo de trabajo completo.

- **`extraer_tablas_ligie`**: Invoca el motor de extracción sobre el PDF, itera por cada tabla detectada por página enviándolas a estandarizar, las concatena en un DataFrame maestro y finalmente ejecuta un barrido general para limpiar artefactos del OCR (filas cortas, encabezados residuales y cuotas fusionadas).

In [2]:
# --- FUNCIONES AUXILIARES ---

def _configurar_entorno(ruta_tesseract):
    """Agrega Tesseract a las variables de entorno del sistema."""
    if os.path.exists(ruta_tesseract):
        os.environ["PATH"] += os.pathsep + ruta_tesseract
        return True
    else:
        print(f"❌ ERROR CRÍTICO: No se encuentra Tesseract en: {ruta_tesseract}")
        return False

def _limpiar_texto(val):
    """Limpia saltos de línea y espacios extra del texto OCR."""
    if pd.isna(val):
        return val
    return str(val).replace('\n', ' ').strip()

def _intentar_separar_cuotas(row):
    """
    Rescata valores fusionados por OCR si Dictamen está vacío 
    e Iniciativa tiene múltiples valores (ej. '35 25').
    """
    iniciativa = str(row['CUOTA_INICIATIVA']).strip()
    dictamen = str(row['CUOTA_DICTAMEN']).strip()
    
    if (dictamen in ['nan', 'None', '']) and ' ' in iniciativa:
        partes = iniciativa.split()
        if len(partes) >= 2:
            nuevo_dictamen = partes[-1]
            nueva_iniciativa = " ".join(partes[:-1])
            return nueva_iniciativa, nuevo_dictamen
            
    return row['CUOTA_INICIATIVA'], row['CUOTA_DICTAMEN']


# --- FUNCIONES INTERMEDIAS ---

def __estandarizar_tabla__(df, num_pagina):
    """
    Limpia estructuralmente una tabla detectada antes de ser concatenada.
    """
    df = df.dropna(how='all')
    df = df.dropna(axis=1, how='all')
    
    # Forzar estructura de 5 columnas
    columnas_actuales = df.shape[1]
    if columnas_actuales > 5:
        df = df.iloc[:, :5]
    elif columnas_actuales < 5:
        for _ in range(5 - columnas_actuales):
            df[len(df.columns)] = None

    # Aplicar nombres estándar
    df.columns = COLUMNAS_ESTANDAR
    
    # Eliminar fila si resulta ser un encabezado escaneado como dato
    if not df.empty:
        fila_0 = " ".join([str(x).upper() for x in df.iloc[0].values])
        if "CÓDIGO" in fila_0 or "DESCRIPCIÓN" in fila_0:
            df = df.iloc[1:] 
    
    # Metadato de trazabilidad
    df['Pagina_Origen'] = num_pagina + 1
    return df


# --- FUNCIONES PRINCIPALES ---

def extraer_tablas_ligie(ruta_pdf):
    """
    Orquestador principal: Configura motor OCR, lee PDF, concatena 
    tablas y ejecuta rutinas de limpieza post-extracción.
    """
    try:
        ocr = TesseractOCR(n_threads=1, lang="spa")
    except Exception as e:
        print(f"❌ Error al inicializar OCR: {e}")
        return pd.DataFrame()

    print(f">> 🔄 Leyendo PDF base: {ruta_pdf}...")
    doc = PDF(src=ruta_pdf)

    print("   ⏳ Ejecutando análisis OCR por favor espere...")
    try:
        tablas_extraidas = doc.extract_tables(
            ocr=ocr,
            implicit_rows=False,
            borderless_tables=False,
            min_confidence=50
        )
    except Exception as e:
        print(f"❌ Error crítico al extraer tablas: {e}")
        return pd.DataFrame()

    lista_dfs = []
    # Iterar sobre el diccionario que regresa img2table
    for numero_pagina, tablas in tablas_extraidas.items():
        for tabla in tablas:
            df_limpio = __estandarizar_tabla__(tabla.df, numero_pagina)
            if not df_limpio.empty:
                lista_dfs.append(df_limpio)

    if not lista_dfs:
        print("⚠️ Advertencia: No se extrajeron tablas válidas del documento.")
        return pd.DataFrame()

    print(f"   ✅ Se encontraron {len(lista_dfs)} tablas. Concatenando y limpiando...")
    df_final = pd.concat(lista_dfs, ignore_index=True)
    
    # --- Limpieza Post-Concatenación ---
    # 1. Limpieza de strings
    for col in COLUMNAS_ESTANDAR:
        df_final[col] = df_final[col].apply(_limpiar_texto)
        
    # 2. Filtrado de artefactos y encabezados residuales
    filtro_basura = df_final['CÓDIGO'].str.upper().str.contains("CÓDIGO|DESCRIPCIÓN", na=False)
    df_final = df_final[~filtro_basura]
    
    # 3. Filtrar códigos rotos OCR (Demasiado cortos)
    df_final = df_final[df_final['CÓDIGO'].str.len() > 3]
    
    # 4. Separación final de cuotas fusionadas en una misma celda
    df_final[['CUOTA_INICIATIVA', 'CUOTA_DICTAMEN']] = df_final.apply(
        lambda x: pd.Series(_intentar_separar_cuotas(x)), axis=1
    )

    return df_final

### Fase 1: Ejecución OCR y Procesamiento de Tablas
Se verifica la integridad de las rutas y del entorno Tesseract antes de disparar el orquestador principal de extracción. Este proceso puede tardar dependiendo de los recursos del sistema y la extensión del PDF.

In [3]:
DF_RESULTADO = pd.DataFrame()

if _configurar_entorno(RUTA_TESSERACT):
    if os.path.exists(PATH_PDF_LIGIE):
        DF_RESULTADO = extraer_tablas_ligie(PATH_PDF_LIGIE)
    else:
        print(f"❌ ERROR: No se encontró el archivo PDF en la ruta especificada: {PATH_PDF_LIGIE}")
else:
    print("❌ ERROR: Entorno no configurado correctamente. Deteniendo ejecución.")

### Fase 2: Exportación de Resultados
Se verifica que la extracción haya sido exitosa (DataFrame no vacío) y se procede a guardar el compilado final en formato Excel, capturando posibles errores de permisos si el archivo de destino se encontrara abierto.

In [4]:
if not DF_RESULTADO.empty:
    try:
        print(f"\n>> Generando archivo de salida Excel...")
        DF_RESULTADO.to_excel(PATH_SALIDA_LIGIE, index=False)
        print(f"🎉 ¡ÉXITO! Archivo guardado correctamente en: {PATH_SALIDA_LIGIE}")
        print(f"   Total de registros extraídos consolidados: {len(DF_RESULTADO)}\n")
        print("Vista Previa de Datos:")
        print(DF_RESULTADO.head())
    except PermissionError:
        print(f"❌ ERROR DE PERMISOS: Asegúrate de cerrar el archivo Excel '{PATH_SALIDA_LIGIE}' antes de correr el proceso.")
    except Exception as e:
        print(f"❌ ERROR CRÍTICO AL GUARDAR: {e}")
else:
    print("⚠️ Advertencia: No hay datos para exportar.")